# 教材人物识别代码 v3.2
## 改动说明（相对v3.1）
- API密钥从 `.env` 文件读取，不再硬编码
- 路径改为项目相对路径
- 输出字段统一（去掉枚举类前缀），与下游职业分类代码兼容
- 同时输出**人头计数表**和**独立职业表**（按page+profession去重）

In [ ]:
import os
import json
import base64
import time
import re
from pathlib import Path
from enum import Enum

import fitz  # PyMuPDF
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import List, Optional

# 加载 .env（初始识别阶段的API密钥）
env_path = Path(__file__).parent / ".env" if "__file__" in dir() else Path(".env")
load_dotenv(dotenv_path=env_path)

# API 配置
client = OpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
)
MODEL_NAME = os.getenv("MODEL_NAME", "openai/gpt-5-chat")

# 项目路径（相对于notebook所在目录）
PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "数据"
RESULT_DIR = PROJECT_ROOT / "结果" / "1.识别结果"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print(f"项目根目录: {PROJECT_ROOT}")
print(f"数据目录: {DATA_DIR}")
print(f"结果目录: {RESULT_DIR}")
print(f"模型: {MODEL_NAME}")

## 数据模型定义

In [ ]:
# --- 枚举定义 ---
class GenderEnum(str, Enum):
    MALE = "男"
    FEMALE = "女"
    UNKNOWN = "未知"

class SourceTypeEnum(str, Enum):
    ILLUSTRATION = "插图"
    ILLUSTRATION_AND_TEXT = "插图和文本"

class ScenarioEnum(str, Enum):
    FAMILY = "家庭"
    SCHOOL = "学校"
    WORKPLACE = "工作场所"
    PUBLIC = "公共场所"
    OUTDOOR = "户外"
    OTHER = "其他"

# --- Pydantic 数据模型 ---
class Character(BaseModel):
    identifier: str = Field(description="人物标识：姓名或简短外貌描述")
    profession: str = Field(description="职业（如医生、学生、未知等）")
    gender: GenderEnum = Field(description="性别")
    source_type: SourceTypeEnum = Field(description="判断依据")
    scenario: ScenarioEnum = Field(description="场景类型")

class PageResult(BaseModel):
    page: int = Field(description="页码")
    characters: List[Character] = Field(default_factory=list, description="该页识别到的人物列表")

class BookResult(BaseModel):
    results: List[PageResult] = Field(default_factory=list)

print("数据模型定义完成")

## Prompt 与 PDF 处理函数

In [ ]:
SYSTEM_PROMPT = """你是一个教材插图分析专家。你的任务是识别教材页面图片中出现的所有人物，并提取以下信息：

对于每个人物：
1. identifier: 人物标识——如果能从文本/插图中得知姓名则用姓名，否则用简短外貌描述（如"戴眼镜的中年男性"）
2. profession: 职业——根据插图和文本判断人物的职业或身份（如"医生"、"学生"、"警察"），如果无法判断则填"未知"
3. gender: 性别——"男"、"女"或"未知"
4. source_type: 判断依据——"插图"（仅根据插图判断）或"插图和文本"（结合了文本信息）
5. scenario: 场景类型——"家庭"、"学校"、"工作场所"、"公共场所"、"户外"或"其他"

注意事项：
- 只识别插图中可见的人物，不要凭空想象
- 每个可辨认的人物都要单独记录（包括集体照中的每个人）
- 如果一页中没有人物插图，返回空的characters列表
- 仔细区分不同人物，不要遗漏也不要重复
- 职业判断要结合插图内容和页面文本信息

请以JSON格式返回结果。"""

def pdf_page_to_base64(pdf_path: str, page_num: int, dpi: int = 200) -> str:
    """将PDF的指定页转换为base64编码的图片"""
    doc = fitz.open(pdf_path)
    page = doc[page_num]
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    pix = page.get_pixmap(matrix=mat)
    img_bytes = pix.tobytes("png")
    doc.close()
    return base64.b64encode(img_bytes).decode("utf-8")

def get_pdf_page_count(pdf_path: str) -> int:
    """获取PDF总页数"""
    doc = fitz.open(pdf_path)
    count = len(doc)
    doc.close()
    return count

print("Prompt 和 PDF 处理函数定义完成")

## API 调用与结果处理

In [ ]:
def analyze_page(pdf_path: str, page_num: int, max_retries: int = 3) -> PageResult:
    """调用多模态大模型分析单页PDF"""
    img_b64 = pdf_page_to_base64(pdf_path, page_num)
    page_display = page_num + 1  # 显示用页码从1开始

    # 构建JSON Schema用于structured output
    schema = BookResult.model_json_schema()
    page_schema = PageResult.model_json_schema()

    user_content = [
        {"type": "text", "text": f"这是教材的第{page_display}页。请识别此页中所有人物并提取信息。返回JSON格式，包含page（页码={page_display}）和characters列表。"},
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_b64}"}}
    ]

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_content}
                ],
                response_format={"type": "json_object"},
                temperature=0.1,
                max_tokens=4096,
            )
            content = response.choices[0].message.content
            data = json.loads(content)

            # 兼容不同返回格式
            if "characters" in data:
                chars_raw = data["characters"]
            elif "results" in data and len(data["results"]) > 0:
                chars_raw = data["results"][0].get("characters", [])
            else:
                chars_raw = []

            # 解析每个人物
            characters = []
            for c in chars_raw:
                try:
                    char = Character(
                        identifier=c.get("identifier", "未知"),
                        profession=c.get("profession", "未知"),
                        gender=c.get("gender", "未知"),
                        source_type=c.get("source_type", "插图"),
                        scenario=c.get("scenario", "其他"),
                    )
                    characters.append(char)
                except Exception as e:
                    print(f"  警告: 第{page_display}页某人物解析失败: {e}, 原始数据: {c}")

            result = PageResult(page=page_display, characters=characters)
            print(f"  第{page_display}页: 识别到 {len(characters)} 个人物")
            return result

        except Exception as e:
            print(f"  第{page_display}页 第{attempt+1}次尝试失败: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)

    print(f"  第{page_display}页: 所有重试均失败，返回空结果")
    return PageResult(page=page_display, characters=[])

print("API 调用函数定义完成")

## 结果转换与导出（人头计数表 + 独立职业表）

In [ ]:
def clean_enum_value(val) -> str:
    """清洗枚举值，取中文value"""
    if hasattr(val, 'value'):
        return val.value
    return str(val)

def results_to_dataframes(book_result: BookResult):
    """将识别结果转换为两个DataFrame：人头计数表和独立职业表"""
    rows = []
    for page_result in book_result.results:
        for char in page_result.characters:
            rows.append({
                "page": page_result.page,
                "identifier": char.identifier,
                "profession": char.profession,
                "gender": clean_enum_value(char.gender),
                "source_type": clean_enum_value(char.source_type),
                "scenario": clean_enum_value(char.scenario),
            })

    # 人头计数表：每个人物一条记录
    df_headcount = pd.DataFrame(rows)

    if df_headcount.empty:
        df_unique = pd.DataFrame(columns=["page", "profession", "gender", "source_type", "scenario", "count"])
        return df_headcount, df_unique

    # 独立职业表：按 (page, profession) 去重
    # 保留该组合下的人数count，gender取众数，source_type/scenario取第一个
    df_unique = (
        df_headcount
        .groupby(["page", "profession"], as_index=False)
        .agg(
            gender=("gender", lambda x: x.mode().iloc[0] if not x.mode().empty else "未知"),
            source_type=("source_type", "first"),
            scenario=("scenario", "first"),
            count=("identifier", "count"),
        )
    )

    return df_headcount, df_unique

def save_results(df_headcount: pd.DataFrame, df_unique: pd.DataFrame, pdf_name: str):
    """保存两种表格到Excel"""
    base_name = Path(pdf_name).stem

    path_head = RESULT_DIR / f"{base_name}_人头计数.xlsx"
    path_unique = RESULT_DIR / f"{base_name}_独立职业.xlsx"

    df_headcount.to_excel(path_head, index=False)
    df_unique.to_excel(path_unique, index=False)

    print(f"人头计数表已保存: {path_head}")
    print(f"  共 {len(df_headcount)} 条记录")
    print(f"独立职业表已保存: {path_unique}")
    print(f"  共 {len(df_unique)} 条记录（去重后）")

    return path_head, path_unique

print("结果转换与导出函数定义完成")

## 主执行：选择PDF并运行识别

In [ ]:
# --- 配置区：修改这里选择要处理的PDF ---
PDF_FILENAME = "【人教版】一年级下册(2025春版)道德与法治电子课本.pdf"

# 可选：指定页码范围（None表示处理全部页）
START_PAGE = None  # 起始页码（从1开始），如 START_PAGE = 1
END_PAGE = None    # 结束页码（包含），如 END_PAGE = 10

# --- 执行识别 ---
pdf_path = DATA_DIR / PDF_FILENAME
assert pdf_path.exists(), f"PDF文件不存在: {pdf_path}"

total_pages = get_pdf_page_count(str(pdf_path))
start = (START_PAGE or 1) - 1  # 转为0-indexed
end = END_PAGE or total_pages

print(f"PDF: {PDF_FILENAME}")
print(f"总页数: {total_pages}, 处理范围: 第{start+1}页 ~ 第{end}页")
print(f"{'='*50}")

all_page_results = []
for i in range(start, end):
    result = analyze_page(str(pdf_path), i)
    all_page_results.append(result)

book_result = BookResult(results=all_page_results)

# 转换并保存
df_headcount, df_unique = results_to_dataframes(book_result)
save_results(df_headcount, df_unique, PDF_FILENAME)

print(f"\n{'='*50}")
print("识别完成！")

## 预览结果

In [ ]:
print("=== 人头计数表（前10行）===")
display(df_headcount.head(10))

print(f"\n=== 独立职业表（前10行）===")
display(df_unique.head(10))

print(f"\n--- 统计摘要 ---")
print(f"人头计数表: {len(df_headcount)} 条记录, 涉及 {df_headcount['page'].nunique()} 页")
print(f"独立职业表: {len(df_unique)} 条记录（去重后）")
if not df_headcount.empty:
    print(f"\n性别分布（人头计数）:")
    print(df_headcount["gender"].value_counts().to_string())
    print(f"\n职业TOP10（人头计数）:")
    print(df_headcount["profession"].value_counts().head(10).to_string())